# Conversion for Mobile Deployment


- PyTorch → ONNX
- ONNX → TensorFlow
- TensorFlow → TFLite + INT8 Quantization

References
- https://github.com/DanieliusKr/onnx-example/blob/main/onnx_example.ipynb
- https://www.geeksforgeeks.org/machine-learning/convert-pytorch-model-to-tf-lite-with-onnx-tf/

## Conversion to ONNX

### Import

In [ ]:
try:
    import google.colab
    !pip install onnx
    !pip install onnxruntime
    !pip install onnx-tf
    !pip install tensorflow-addons
    !pip install tensorflow-probability
except ImportError:
    # Running locally. Install the libraries necessary.
    pass

In [ ]:
import torch as torch
import torch.nn as nn
import time
import random
import os

import torch.onnx
import onnxruntime as ort
from onnx_tf.backend import prepare
import tensorflow as tf

### Configurations

In [ ]:
@dataclass
class Config:
    """All hyperparameters for Prototype 9."""
    # Data
    height: int = 128
    width: int = 128
    channels: int = 6
    fps: int = 10
    duration: int = 2
    seq_len: int = 20  # fps * duration

    # Training
    batch_size: int = 2
    accumulation_steps: int = 4  # Effective batch = 6 * 4 = 24
    num_epochs: int = 5
    learning_rate: float = 1e-4
    seed: int = 8

    # Early Stopping
    early_stop_patience: int = 5
    min_delta: float = 0.01  # 0.01% improvement threshold

    # Regularization
    dropout_rate: float = 0.5
    max_grad_norm: float = 1.0

    # LR Scheduler
    lr_factor: float = 0.5
    lr_patience: int = 3
    lr_min: float = 1e-7

    # Model Architecture
    input_dim: int = 6
    hidden_dim: List[int] = None  # [64, 32]
    kernel_size: Tuple[int, int] = (3, 3)
    num_layers: int = 2
    num_classes: int = 3

    # Cache
    reserve_gb: float = 10.0
    eviction_check_interval: int = 10
    eviction_buffer_percent: float = 0.10

    def __post_init__(self):
        if self.hidden_dim is None:
            self.hidden_dim = [64, 32]
        self.seq_len = self.fps * self.duration

# Global config instance
CONFIG = Config()

# Convenience aliases for backward compatibility
HEIGHT = CONFIG.height
WIDTH = CONFIG.width
CHANNELS = CONFIG.channels
FPS = CONFIG.fps
DURATION = CONFIG.duration
SEQ_LEN = CONFIG.seq_len

### ConvLSTM

In [ ]:
# ConvLSTM Architecture for Video Direction Classification
#
# MODEL REQUIREMENTS:
# -------------------
# Input Shape:  [Batch, Seq_Len, Channels, Height, Width]
#               [B, 30, 6, 128, 128] for this config
#
# LAYER STRUCTURE (Config: num_layers=2, hidden_dim=[64, 32]):
# -------------------
# Layer 1: ConvLSTMCell
#   - Input channels:  3 (RGB frames) + 3 (intent)
#   - Hidden channels: 64
#   - Kernel size:     3×3
#   - Output:          [B, 30, 64, 128, 128]
#
# Layer 2: ConvLSTMCell
#   - Input channels:  64 (from Layer 1)
#   - Hidden channels: 32
#   - Kernel size:     3×3
#   - Output:          [B, 30, 32, 128, 128]
#
# Classification Head:
#   - Takes last time step: [B, 32, 128, 128]
#   - Flattens to:          [B, 524288]  (32 × 128 × 128)
#   - Dropout (0.5)
#   - Linear layer:         [524288 → 3]
#   - Output logits:        [B, 3] (Front, Left, Right)
#
# KERNEL OPERATIONS:
# -------------------
# Each ConvLSTMCell uses 3×3 convolutions with:
#   - Padding: 1 (preserves spatial dimensions)
#   - 4 parallel convolutions for LSTM gates: [input, forget, output, cell]
#   - Total parameters per cell: (input_dim + hidden_dim) × 4 × hidden_dim × 3 × 3
#
# Example Layer 1: (6 + 64) × 4 × 64 × 9 = 161,280 parameters
# Example Layer 2: (64 + 32) × 4 × 32 × 9 = 110,592 parameters

class ConvLSTMCell(nn.Module):
    """
    Single ConvLSTM memory unit - processes one frame at a time.

    Combines convolutional feature extraction with LSTM temporal memory.
    Uses 4 gates (input, forget, output, cell) to control information flow.

    Args:
        input_dim: Number of input channels (e.g., 6 for RGB and intent, 64 for hidden layer)
        hidden_dim: Number of hidden state channels (e.g., 64, 32)
        kernel_size: Size of convolutional kernel (e.g., (3, 3))
        bias: Whether to use bias in convolutions
    """

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        kernel_size: Tuple[int, int],
        bias: bool = True
    ) -> None:
        super(ConvLSTMCell, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.kernel_size = kernel_size
        self.padding = (kernel_size[0] // 2, kernel_size[1] // 2)
        self.bias = bias

        self.conv = nn.Conv2d(
            in_channels=self.input_dim + self.hidden_dim,
            out_channels=4 * self.hidden_dim,
            kernel_size=self.kernel_size,
            padding=self.padding,
            bias=self.bias
        )

    def forward(
        self,
        input_tensor: torch.Tensor,
        cur_state: Tuple[torch.Tensor, torch.Tensor]
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Process one frame through ConvLSTM cell.

        LSTM Gate Equations:
        --------------------
        i_t = σ(W_xi * x_t + W_hi * h_{t-1})  ← Input gate (what to add)
        f_t = σ(W_xf * x_t + W_hf * h_{t-1})  ← Forget gate (what to keep)
        o_t = σ(W_xo * x_t + W_ho * h_{t-1})  ← Output gate (what to expose)
        g_t = tanh(W_xg * x_t + W_hg * h_{t-1}) ← Cell candidate

        c_t = f_t ⊙ c_{t-1} + i_t ⊙ g_t        ← Update cell state
        h_t = o_t ⊙ tanh(c_t)                   ← Update hidden state

        where ⊙ = element-wise multiplication, σ = sigmoid
        """
        h_cur, c_cur = cur_state  # Extract previous hidden and cell states

        # Concatenate input frame with previous hidden state
        # [B, input_dim, H, W] + [B, hidden_dim, H, W] → [B, input_dim+hidden_dim, H, W]
        combined = torch.cat([input_tensor, h_cur], dim=1)

        # Apply convolution to produce 4 gate activations
        # [B, input_dim+hidden_dim, H, W] → [B, 4×hidden_dim, H, W]
        combined_conv = self.conv(combined)

        # Split into 4 separate gates, each with hidden_dim channels
        # [B, 4×hidden_dim, H, W] → 4 tensors of [B, hidden_dim, H, W]
        cc_i, cc_f, cc_o, cc_g = torch.split(combined_conv, self.hidden_dim, dim=1)

        # Apply activation functions to each gate
        i = torch.sigmoid(cc_i)  # Input gate: how much new info to add (0-1)
        f = torch.sigmoid(cc_f)  # Forget gate: how much old info to keep (0-1)
        o = torch.sigmoid(cc_o)  # Output gate: how much to expose (0-1)
        g = torch.tanh(cc_g)     # Cell candidate: new information (-1 to 1)

        # Update cell state: forget old + add new
        c_next = f * c_cur + i * g

        # Update hidden state: apply output gate to cell state
        h_next = o * torch.tanh(c_next)

        return h_next, c_next

    def init_hidden(
        self,
        batch_size: int,
        image_size: Tuple[int, int]
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Initialize hidden and cell states with zeros.

        Returns:
            h_0: [batch_size, hidden_dim, height, width] - initial hidden state
            c_0: [batch_size, hidden_dim, height, width] - initial cell state

        Example for Layer 1: [B, 64, 128, 128]
        Example for Layer 2: [B, 32, 128, 128]
        """
        height, width = image_size
        device = self.conv.weight.device  # Match device of model parameters
        return (
            torch.zeros(batch_size, self.hidden_dim, height, width, device=device),
            torch.zeros(batch_size, self.hidden_dim, height, width, device=device)
        )


class ConvLSTM(nn.Module):
    """
    Multi-layer ConvLSTM - processes video sequences frame by frame.

    Stacks multiple ConvLSTMCell layers to form a deep temporal network.
    Each layer processes all frames sequentially, then passes to next layer.

    FORWARD PASS FLOW (num_layers=2, seq_len=30):
    ------------------------------------------------
    Input: [B, 30, 3, 128, 128]
      ↓
    Layer 1 processes frame-by-frame:
      ├─ Frame 1:  [B, 3, 128, 128] → h1, c1 [B, 64, 128, 128]
      ├─ Frame 2:  [B, 3, 128, 128] → h2, c2 [B, 64, 128, 128]
      ⋮
      └─ Frame 30: [B, 3, 128, 128] → h30, c30 [B, 64, 128, 128]
    Stack outputs → [B, 30, 64, 128, 128]
      ↓
    Layer 2 processes frame-by-frame:
      ├─ Frame 1:  [B, 64, 128, 128] → h1, c1 [B, 32, 128, 128]
      ├─ Frame 2:  [B, 64, 128, 128] → h2, c2 [B, 32, 128, 128]
      ⋮
      └─ Frame 30: [B, 64, 128, 128] → h30, c30 [B, 32, 128, 128]
    Stack outputs → [B, 30, 32, 128, 128]
      ↓
    Final output: Last layer's all time steps [B, 30, 32, 128, 128]

    Args:
        input_dim: Input channels (3 for RGB)
        hidden_dim: List of hidden dims per layer [64, 32]
        kernel_size: Kernel size for conv operations (3, 3)
        num_layers: Number of stacked ConvLSTM layers (2)
        batch_first: If True, input shape is [B, T, C, H, W]
        bias: Use bias in convolutions
        return_all_layers: If True, return outputs from all layers
    """

    def __init__(
        self,
        input_dim: int,               # 3 (RGB channels)
        hidden_dim: List[int],        # [64, 32] - one per layer
        kernel_size: Tuple[int, int], # (3, 3)
        num_layers: int,              # 2
        batch_first: bool = True,
        bias: bool = True,
        return_all_layers: bool = False
    ) -> None:
        super(ConvLSTM, self).__init__()

        # Validate and extend parameters for all layers
        self._check_kernel_size_consistency(kernel_size)
        kernel_size = self._extend_for_multilayer(kernel_size, num_layers)  # [(3,3), (3,3)]
        hidden_dim = self._extend_for_multilayer(hidden_dim, num_layers)    # [64, 32]

        self.input_dim = input_dim          # 3
        self.hidden_dim = hidden_dim        # [64, 32]
        self.kernel_size = kernel_size      # [(3,3), (3,3)]
        self.num_layers = num_layers        # 2
        self.batch_first = batch_first
        self.bias = bias
        self.return_all_layers = return_all_layers

        # Build list of ConvLSTM cells (one per layer)
        # Layer 0: input_dim=3, hidden_dim=64
        # Layer 1: input_dim=64, hidden_dim=32
        cell_list = []
        for i in range(self.num_layers):
            cur_input_dim = self.input_dim if i == 0 else self.hidden_dim[i - 1]
            cell_list.append(ConvLSTMCell(
                input_dim=cur_input_dim,
                hidden_dim=self.hidden_dim[i],
                kernel_size=self.kernel_size[i],
                bias=self.bias
            ))
        self.cell_list = nn.ModuleList(cell_list)

    def forward(
        self,
        input_tensor: torch.Tensor,  # [B, 30, 3, 128, 128]
        hidden_state: Optional[List[Tuple[torch.Tensor, torch.Tensor]]] = None
    ) -> Tuple[List[torch.Tensor], List[Tuple[torch.Tensor, torch.Tensor]]]:
        """
        Forward pass through all ConvLSTM layers.

        Process:
        --------
        1. For each layer (2 layers):
           2. Initialize hidden states if not provided
           3. Process all 30 frames sequentially
           4. Stack frame outputs → [B, 30, hidden_dim, H, W]
           5. Use this as input to next layer

        Returns:
        --------
        layer_output_list: List of outputs from each layer
                          [[B, 30, 32, 128, 128]] if return_all_layers=False
                          [[B, 30, 64, 128, 128], [B, 30, 32, 128, 128]] if True

        last_state_list: List of final (h, c) for each layer
                        [(h_layer1, c_layer1), (h_layer2, c_layer2)]
        """
        # Convert to batch_first format if needed
        if not self.batch_first:
            # [T, B, C, H, W] → [B, T, C, H, W]
            input_tensor = input_tensor.permute(1, 0, 2, 3, 4)

        # Extract dimensions
        b, seq_len, _, h, w = input_tensor.size()  # [B=6, T=30, C=3, H=128, W=128]

        # Initialize hidden states for all layers if not provided
        if hidden_state is None:
            hidden_state = self._init_hidden(batch_size=b, image_size=(h, w))

        layer_output_list = []
        last_state_list = []
        cur_layer_input = input_tensor  # Start with raw video [B, 30, 3, 128, 128]

        # Process each layer sequentially
        for layer_idx in range(self.num_layers):
            h, c = hidden_state[layer_idx]  # Get initial states for this layer
            output_inner = []  # Store outputs for all time steps

            # Process each frame (time step) sequentially
            # This is where the RECURRENCE happens
            for t in range(seq_len):
                # Extract frame t: [B, C, H, W]
                input_t = cur_layer_input[:, t, :, :, :]

                # Process through ConvLSTM cell
                # Input: [B, C_in, H, W] + previous (h, c)
                # Output: new (h, c) with shape [B, hidden_dim, H, W]
                h, c = self.cell_list[layer_idx](input_tensor=input_t, cur_state=[h, c])

                # Save hidden state for this time step
                output_inner.append(h)

            # Stack all time steps: list of [B, hidden_dim, H, W] → [B, T, hidden_dim, H, W]
            layer_output = torch.stack(output_inner, dim=1)

            # This becomes input to next layer
            cur_layer_input = layer_output

            layer_output_list.append(layer_output)
            last_state_list.append([h, c])

        # If only returning final layer output (default)
        if not self.return_all_layers:
            layer_output_list = layer_output_list[-1:]  # Keep only last layer
            last_state_list = last_state_list[-1:]

        return layer_output_list, last_state_list

    def _init_hidden(
        self,
        batch_size: int,
        image_size: Tuple[int, int]
    ) -> List[Tuple[torch.Tensor, torch.Tensor]]:
        """Initialize hidden states for all layers."""
        return [self.cell_list[i].init_hidden(batch_size, image_size)
                for i in range(self.num_layers)]

    @staticmethod
    def _check_kernel_size_consistency(kernel_size: Any) -> None:
        """Validate kernel_size is tuple or list of tuples."""
        if not (isinstance(kernel_size, tuple) or
                (isinstance(kernel_size, list) and all(isinstance(elem, tuple) for elem in kernel_size))):
            raise ValueError('`kernel_size` must be tuple or list of tuples')

    @staticmethod
    def _extend_for_multilayer(param: Any, num_layers: int) -> List:
        """
        Extend parameter for all layers if single value provided.

        Example:
            (3, 3) with num_layers=2 → [(3, 3), (3, 3)]
            [64, 32] with num_layers=2 → [64, 32] (unchanged)
        """
        if not isinstance(param, list):
            param = [param] * num_layers
        return param


class ConvLSTMModel(nn.Module):
    """
    ConvLSTM classifier - processes video and outputs class prediction.

    FULL PIPELINE:
    --------------
    Input Video: [B, 30, 3, 128, 128]
        ↓
    ConvLSTM (2 layers):
        Layer 1: [B, 30, 3, 128, 128] → [B, 30, 64, 128, 128]
        Layer 2: [B, 30, 64, 128, 128] → [B, 30, 32, 128, 128]
        ↓
    Take Last Time Step: [B, 30, 32, 128, 128] → [B, 32, 128, 128]
        ↓
    Global Average Pooling: [B, 32, 128, 128] → [B, 32, 1, 1]
        ↓
    Flatten: [B, 32, 1, 1] → [B, 32]
        ↓
    Dropout(0.5): [B, 32] → [B, 32]
        ↓
    Linear: [B, 32] → [B, 3]
        ↓
    Output Logits: [B, 3] (Front=0, Left=1, Right=2)

    PARAMETERS (Optimized for Mobile):
    -----------
    ConvLSTM layers: ~265K parameters
    Linear layer:    32 × 3 = 96 parameters
    Total:           ~265K parameters (vs ~1.84M in original)

    MOBILE DEPLOYMENT BENEFITS:
    ---------------------------
    ✓ 80% smaller model size (better for app bundles)
    ✓ 10-20% faster inference (less computation)
    ✓ Lower memory footprint (better for edge devices)
    ⚠ Expected accuracy drop: 2-5% (acceptable trade-off)

    Args:
        input_dim: Input channels (3 for RGB)
        hidden_dim: Hidden dims per layer [64, 32]
        kernel_size: Conv kernel size (3, 3)
        num_layers: Number of ConvLSTM layers (2)
        height: Frame height (128)
        width: Frame width (128)
        num_classes: Output classes (3)
        dropout_rate: Dropout probability (0.5)
    """

    def __init__(
        self,
        input_dim: int,               # 3 (RGB)
        hidden_dim: List[int],        # [64, 32]
        kernel_size: Tuple[int, int], # (3, 3)
        num_layers: int,              # 2
        height: int,                  # 128
        width: int,                   # 128
        batch_first: bool = True,
        bias: bool = True,
        return_all_layers: bool = False,
        num_classes: int = 3,        # Front, Left, Right
        dropout_rate: float = 0.5
    ) -> None:
        super(ConvLSTMModel, self).__init__()

        # Build the ConvLSTM backbone
        self.convlstm = ConvLSTM(
            input_dim, hidden_dim, kernel_size, num_layers,
            batch_first=batch_first, bias=bias,
            return_all_layers=return_all_layers
        )

        # Global Average Pooling
        # Reduces spatial dimensions [B, C, H, W] → [B, C, 1, 1]
        # Eliminates 99.99% of FC layer parameters (1,572,867 → 99)
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))

        # Dropout for regularization (prevents overfitting)
        self.dropout = nn.Dropout(p=dropout_rate)

        # Classification head: maps pooled features to class logits
        # Input size: hidden_dim[-1] = 32 (after GAP reduces [32, 128, 128] to [32, 1, 1])
        # Output size: num_classes = 3
        # Parameters: 32 × 3 + 3 = 99 (vs 1,572,867 in original model)
        # Model size reduction: ~7.4 MB → ~1.5 MB (-80%)
        self.linear = nn.Linear(hidden_dim[-1], num_classes)

    def forward(
        self,
        input_tensor: torch.Tensor,  # [B, 30, 3, 128, 128]
        hidden_state: Optional[List[Tuple[torch.Tensor, torch.Tensor]]] = None
    ) -> torch.Tensor:
        """
        Full forward pass: video → ConvLSTM → classification.

        Steps:
        ------
        1. Process video through ConvLSTM layers
           [B, 30, 3, 128, 128] → [B, 30, 32, 128, 128]

        2. Take last time step (frame 30 contains all temporal info)
           [B, 30, 32, 128, 128] → [B, 32, 128, 128]

        3. Apply Global Average Pooling (Mobile Optimization)
           [B, 32, 128, 128] → [B, 32, 1, 1]
           Aggregates spatial information into channel-wise features

        4. Flatten pooled features
           [B, 32, 1, 1] → [B, 32]

        5. Apply dropout (training only)
           [B, 32] → [B, 32]

        6. Linear classification layer
           [B, 32] → [B, 3]

        Returns:
        --------
        logits: [B, 3] - raw scores for each class (before softmax)
        """
        # Process through ConvLSTM layers
        x, _ = self.convlstm(input_tensor)  # x is list, x[0] = [B, 30, 32, 128, 128]

        # Extract last time step (final frame has seen all previous frames)
        last_time_step = x[0][:, -1, :, :, :]  # [B, 32, 128, 128]

        # Apply Global Average Pooling to reduce spatial dimensions
        # [B, 32, 128, 128] → [B, 32, 1, 1]
        # This aggregates spatial information into channel-wise features
        pooled = self.global_avg_pool(last_time_step)

        # Flatten pooled features for classification
        # [B, 32, 1, 1] → [B, 32]
        flattened = torch.flatten(pooled, start_dim=1)

        # Apply dropout (active during training, disabled during eval)
        flattened = self.dropout(flattened)

        # Final linear layer: produce class logits
        return self.linear(flattened)  # [B, 3]

### Model

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the trained model
model = ConvLSTMModel(
    input_dim=CONFIG.input_dim,
    hidden_dim=CONFIG.hidden_dim,
    kernel_size=CONFIG.kernel_size,
    num_layers=CONFIG.num_layers,
    height=HEIGHT,
    width=WIDTH,
    num_classes=CONFIG.num_classes,
    dropout_rate=CONFIG.dropout_rate
).to(DEVICE)

parent_dir = os.path.dirname(os.getcwd())
best_model_path = os.path.join(parent_dir, "best_convlstm.pth")
model.load_state_dict(torch.load(best_model_path))

model.eval()

def get_input() -> torch.Tensor:
    """ Generate random input data with the same shape as the model's expected input """
    # Generate random video
    video_tensor = torch.rand(1, 3, HEIGHT, WIDTH).to(self.device)
    # Generate random intent
    intent_direction = random.randint(1, 3)
    # Generate random intent position
    intent_position = random.randint(0, 9)

    # Convert random video to frames
    frames = []
    for i in range(video_tensor.shape[0]):
        frame_tensor = video_tensor[i]
        
        intent_torch = torch.zeros((3, HEIGHT, WIDTH))
        # If intent exists, add intent in its intent position for 1 second (10 frames)
        if intent_position <= i and (intent_position + 10) > i:
            # Convert intent to one-hot vector by filling the specified channel with 1
            intent_torch[intent_direction, :, :] = 1

        # Append the intent as a channel to the video frame
        frame_tensor = torch.cat((frame_tensor, intent_torch), dim=0)
        frames.append(frame_tensor)

    video_tensor = torch.stack(frames, dim=0)
    return video_tensor

# Input for inference
input = get_input()

# Perform inference
output = model(input)
print("PyTorch Inference Output:", output.detach().numpy())

### Conversion to ONNX

In [ ]:
onnx_path = "convlstm.onnx"

# Convert from PyTorch to ONNX
torch.onnx.export(
    model,                         
    input,                        
    onnx_path,                      
    input_names=["input"],          
    output_names=["output"],        
)

print(f"Success: conversion to ONNX. File is saved at: {onnx_path}")

### Loads ONNX

In [ ]:
# Load the ONNX model
onnx_path = "convlstm.onnx"
ort_session = ort.InferenceSession(onnx_path)

# Prepare sample input data (same shape as the PyTorch model)
onnx_input = get_input()

# Run inference on the ONNX model
onnx_output = ort_session.run(None, {"input": onnx_input})
print(f"ONNX Inference Output: {onnx_output}")

### Comparison of Speed

In [ ]:
# PyTorch inference time
total_time = 0
iterations = 50

for i in range(iterations):
  input = get_input()
  start_time = time.time()
  output = model(input)
  end_time = time.time()
  total_time += (end_time - start_time)

print(f"Total time: {round(total_time/iterations, 2)} seconds")

# ONNX inference time
total_time = 0
iterations = 50
onnx_path = "convlstm.onnx"
ort_session = ort.InferenceSession(onnx_path)

for i in range(iterations):
  input = get_input()
  start_time = time.time()
  output = ort_session.run(None, {"input": input})
  end_time = time.time()
  total_time += (end_time - start_time)

print(f"Total time: {round(total_time/iterations, 2)} seconds")

## Conversion to TensorFlow

In [ ]:
onnx_model = onnx.load(onnx_path)
tf_model = prepare(onnx_model)

tf_dir = "tf_model"
os.makedirs(tf_dir, exist_ok=True)

# Convert from ONNX to Tensorflow
tf_model.export_graph(tf_dir)
print(f"Success: conversion to Tensorflow. File is saved at: {tf_dir}")

## Conversion to TFLite

In [ ]:
""" Convert from Tensorflow to TFLite """
converter = tf.lite.TFLiteConverter.from_saved_model(tf_dir)
# Dynamic Range Quantization: INT 8 Quantization
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

# Save the TFLite model
with open ("convlstm.tflite", "wb") as f:
    f.write(tflite_model)

print(f"Success: conversion to TFLite. File is saved at: {os.path.join(tf_dir, "convlstm.tflite")}")